# 08 — Training-cap comparison across FD001–FD004
Run this **new experiment** after the earlier notebooks; previous outputs are preserved.

Test training caps 75, 100, 125, 150, 175, 200 and uncapped. Select the cap separately for each dataset using mean **uncapped RUL RMSE** across engine-separated validation snapshots. All candidates see identical splits, snapshots, features and RF settings. This notebook isolates the training cap: it does not retune feature variants.

Raw RMSE is the primary score; MAE, R², bias and a common-100 diagnostic are secondary. A smaller error against a different capped target is not evidence of improved lifetime prediction. Existing official test results have already been inspected in this project, so further test results are exploratory rather than a pristine untouched holdout.

**Requirements:** numpy, pandas, matplotlib, scikit-learn, joblib. Put NASA files in `data/raw/`. Run all cells in order. Defaults require 4 × 3 × 7 = 84 validation fits, plus final fits. Start with one dataset if needed. Full NASA results are not precomputed here.


In [ ]:
from pathlib import Path
import json, hashlib, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
try:
    from IPython.display import display
except ImportError:
    display = print

DATASETS = ["FD001", "FD002", "FD003", "FD004"]
CAPS = [75, 100, 125, 150, 175, 200, None]  # None = uncapped
SPLIT_SEEDS = [42, 52, 62]
N_ESTIMATORS = 150  # use 50 for a preliminary run; rerun all for final settings
RANDOM_STATE = 42
ROLLING_WINDOW = 5
VARIANT = "Improved RF"  # fixed across caps; same temporal features as notebook 04
RUN_OFFICIAL_TEST = True  # test labels are loaded only after choices are saved
HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name.lower() == "notebooks" else HERE
DATA_DIR = ROOT / "data" / "raw"
OUT = ROOT / "results" / "training_cap_experiment"
OUT.mkdir(parents=True, exist_ok=True)
setting_cols = [f"setting_{i}" for i in range(1, 4)]
sensor_cols = [f"sensor_{i}" for i in range(1, 22)]
columns = ["engine_id", "cycle"] + setting_cols + sensor_cols
config = dict(datasets=DATASETS, caps=CAPS, split_seeds=SPLIT_SEEDS,
              n_estimators=N_ESTIMATORS, variant=VARIANT, window=ROLLING_WINDOW,
              random_state=RANDOM_STATE, selection="mean validation raw_RMSE")
signature = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
RUN = OUT / signature
RUN.mkdir(exist_ok=True)
(RUN / "config.json").write_text(json.dumps(
    dict(**config, sklearn=sklearn.__version__, python=platform.python_version()), indent=2))
print("Output:", RUN)


In [ ]:
def load_data(ds, kind):
    path = DATA_DIR / f"{kind}_{ds}.txt"
    df = pd.read_csv(path, sep=r"\s+", header=None)
    if kind == "RUL":
        assert df.shape[1] == 1
        return df.iloc[:, 0].to_numpy(float)
    assert df.shape[1] == len(columns), f"Unexpected columns: {path}"
    df.columns = columns
    assert not df[["engine_id", "cycle"]].duplicated().any()
    assert np.isfinite(df.to_numpy(float)).all(), "Expected finite NASA input"
    df = df.sort_values(["engine_id", "cycle"]).reset_index(drop=True)
    if kind == "train":
        df["RUL_raw"] = df.groupby("engine_id").cycle.transform("max") - df.cycle
    return df

def cap_label(cap):
    return "uncapped" if cap is None else str(int(cap))

def capped_target(y, cap):
    a = np.asarray(y, dtype=float)
    return a if cap is None else np.minimum(a, cap)

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestRegressor(n_estimators=N_ESTIMATORS,
         max_depth=None, min_samples_leaf=4, max_features="sqrt",
         n_jobs=-1, random_state=RANDOM_STATE))])

def score(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    assert y.shape == p.shape and np.isfinite(p).all()
    e = p-y
    near = y <= 30
    ce = np.minimum(p, 100) - np.minimum(y, 100)
    return dict(raw_RMSE=float(np.sqrt(np.mean(e**2))),
        raw_MAE=float(np.mean(abs(e))), raw_R2=float(r2_score(y,p)) if len(y)>1 and np.var(y)>0 else np.nan,
        raw_bias=float(e.mean()), common100_RMSE=float(np.sqrt(np.mean(ce**2))),
        near_MAE=float(np.mean(abs(e[near]))) if near.any() else np.nan,
        near_bias=float(e[near].mean()) if near.any() else np.nan,
        near_n=int(near.sum()), n=len(y))

def snapshots(val, seed):
    rng = np.random.default_rng(seed)
    cutoffs = {}
    for engine, g in val.groupby("engine_id"):
        end = int(g.cycle.max())
        cutoffs[engine] = int(rng.integers(max(1, int(np.ceil(.4*end))), end+1))
    # Truncate before computing temporal features. RUL_raw retains true failure time.
    partial = val.loc[val.cycle <= val.engine_id.map(cutoffs)].copy()
    return partial


In [ ]:
def add_temporal_features(df, sensors, add_trends=False):
    df = df.sort_values(["engine_id", "cycle"]).copy()
    group = df.groupby("engine_id", sort=False)
    new_cols = {}
    for s in sensors:
        new_cols[f"{s}_rolling_mean"] = group[s].transform(
            lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).mean())
        new_cols[f"{s}_rolling_std"] = group[s].transform(
            lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).std()).fillna(0)
        new_cols[f"{s}_delta"] = group[s].diff().fillna(0)
        if add_trends:
            new_cols[f"{s}_slope"] = group[s].transform(
                lambda x: (x - x.shift(ROLLING_WINDOW - 1)) /
                np.minimum(np.arange(len(x)), ROLLING_WINDOW - 1).clip(min=1)).fillna(0)
            new_cols[f"{s}_change_from_start"] = df[s] - group[s].transform("first")
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


def temporal_columns(sensors, add_trends=False):
    suffixes = ["rolling_mean", "rolling_std", "delta"]
    if add_trends:
        suffixes += ["slope", "change_from_start"]
    return [f"{s}_{suffix}" for s in sensors for suffix in suffixes]


In [ ]:
def add_regimes(fit_df, other_df, sensors, n_regimes=6):
    scaler = StandardScaler().fit(fit_df[setting_cols])
    fit_settings = scaler.transform(fit_df[setting_cols])
    other_settings = scaler.transform(other_df[setting_cols])
    k = min(n_regimes, len(fit_df))
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(fit_settings)
    fit = fit_df.copy()
    other = other_df.copy()
    fit["operating_regime"] = kmeans.predict(fit_settings)
    other["operating_regime"] = kmeans.predict(other_settings)
    for s in sensors:
        stats = fit.groupby("operating_regime")[s].agg(["mean", "std"])
        overall_std = float(fit[s].std())
        if not np.isfinite(overall_std) or overall_std < 1e-8:
            overall_std = 1.0
        means_fit = fit["operating_regime"].map(stats["mean"])
        means_other = other["operating_regime"].map(stats["mean"])
        stds_fit = fit["operating_regime"].map(stats["std"]).fillna(overall_std).clip(lower=1e-8)
        stds_other = other["operating_regime"].map(stats["std"]).fillna(overall_std).clip(lower=1e-8)
        fit[f"{s}_regime_z"] = (fit[s] - means_fit) / stds_fit
        other[f"{s}_regime_z"] = (other[s] - means_other) / stds_other
    return fit, other


def build_features(fit_raw, other_raw, variant):
    # Sensor selection is based on fit partition only.
    sensors = [s for s in sensor_cols if fit_raw[s].std() > 1e-8]
    add_trends = variant == "RF + trends"
    fit = add_temporal_features(fit_raw, sensors, add_trends=add_trends)
    other = add_temporal_features(other_raw, sensors, add_trends=add_trends)
    features = ["cycle"] + setting_cols + sensors + temporal_columns(sensors, add_trends)
    if variant == "RF + regimes":
        fit, other = add_regimes(fit, other, sensors)
        features += ["operating_regime"] + [f"{s}_regime_z" for s in sensors]
    return fit, other, features


## Validate every cap on identical held-out engines
The observation cutoff is sampled between 40% and 100% of each held-out engine's life. Only its last visible observation is scored. No held-out engine contributes training rows or feature fitting. Split averages describe sensitivity to these splits, not independent confidence intervals.

The cap is selected using raw RUL error, so a larger cap is allowed to win if it predicts long remaining lifetimes better. The common-100 diagnostic clips **both predictions and truth** to 100; this convention is explicit and is not used for selection.


In [ ]:
train_sets = {ds: load_data(ds, "train") for ds in DATASETS}
records, predictions = [], []
for ds, train in train_sets.items():
    for seed in SPLIT_SEEDS:
        split = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed)
        fi, vi = next(split.split(train, groups=train.engine_id))
        fit_raw, val_raw = train.iloc[fi].copy(), train.iloc[vi].copy()
        assert set(fit_raw.engine_id).isdisjoint(set(val_raw.engine_id))
        partial = snapshots(val_raw, seed)
        fit, val, features = build_features(fit_raw, partial, VARIANT)
        last = val.groupby("engine_id", as_index=False).tail(1).sort_values("engine_id")
        for cap in CAPS:
            start = time.perf_counter()
            model = make_model().fit(fit[features], capped_target(fit.RUL_raw, cap))
            p = np.maximum(0, model.predict(last[features]))
            records.append(dict(dataset=ds, seed=seed, training_cap=cap_label(cap),
                                fit_seconds=time.perf_counter()-start, **score(last.RUL_raw,p)))
            predictions.append(pd.DataFrame(dict(dataset=ds, seed=seed,
                training_cap=cap_label(cap), engine_id=last.engine_id.to_numpy(),
                observed_cycle=last.cycle.to_numpy(), actual_RUL=last.RUL_raw.to_numpy(),
                predicted_RUL=p)))
            print(ds, seed, cap_label(cap), round(records[-1]["raw_RMSE"], 3), flush=True)
        pd.DataFrame(records).to_csv(RUN / "validation_by_split.csv", index=False)
validation = pd.DataFrame(records)
pd.concat(predictions, ignore_index=True).to_csv(RUN / "validation_predictions.csv", index=False)
summary = validation.groupby(["dataset","training_cap"], as_index=False).agg(
    raw_RMSE=("raw_RMSE","mean"), raw_RMSE_sd=("raw_RMSE","std"),
    raw_MAE=("raw_MAE","mean"), raw_R2=("raw_R2","mean"),
    raw_bias=("raw_bias","mean"), common100_RMSE=("common100_RMSE","mean"),
    fit_seconds=("fit_seconds","mean"))
summary = summary.sort_values(["dataset","raw_RMSE","training_cap"])
choices = summary.groupby("dataset", as_index=False).first()
summary.to_csv(RUN / "validation_summary.csv", index=False)
choices.to_csv(RUN / "selected_caps.csv", index=False)
display(summary.round(3))
print("SELECTED BY VALIDATION RAW RMSE:")
display(choices.round(3))


In [ ]:
labels = [cap_label(c) for c in CAPS]
fig, axes = plt.subplots(2,2,figsize=(12,8),squeeze=False)
for ax, ds in zip(axes.flat, DATASETS):
    d = summary.loc[summary.dataset.eq(ds)].set_index("training_cap").reindex(labels)
    ax.errorbar(labels, d.raw_RMSE, yerr=d.raw_RMSE_sd.fillna(0), marker="o", capsize=3)
    ax.set(title=ds, xlabel="Training cap (cycles)", ylabel="Validation raw-RUL RMSE (cycles)")
    ax.grid(alpha=.2)
for ax in list(axes.flat)[len(DATASETS):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig(RUN / "validation_cap_comparison.png", dpi=200)
plt.show()


## Final official-test comparison
Refit the validation-selected cap and the fixed 100-cycle reference using identical full training data and model settings. Test labels do not choose the cap. If validation selected 100, the two methods are identical and only one fit is needed. Do not choose another cap after inspecting this table.

These are new experimental models; they do not automatically replace the previous report's feature-selected models or results.


In [ ]:
official_rows, official_predictions = [], []
if RUN_OFFICIAL_TEST:
    for ds in DATASETS:
        chosen = choices.loc[choices.dataset.eq(ds), "training_cap"].iloc[0]
        test_raw = load_data(ds, "test")
        truth = load_data(ds, "RUL")
        train = train_sets[ds]
        fit, test, features = build_features(train, test_raw, VARIANT)
        last = test.groupby("engine_id", as_index=False).tail(1).sort_values("engine_id")
        assert np.array_equal(last.engine_id.to_numpy(), np.arange(1,len(truth)+1))
        cache = {}
        for method, label in [("fixed100", "100"), ("validation_selected", chosen)]:
            cap = None if label == "uncapped" else int(label)
            if label not in cache:
                model = make_model().fit(fit[features], capped_target(fit.RUL_raw, cap))
                cache[label] = np.maximum(0, model.predict(last[features]))
            p = cache[label]
            official_rows.append(dict(dataset=ds, method=method, training_cap=label, **score(truth,p)))
            official_predictions.append(pd.DataFrame(dict(dataset=ds, method=method,
                training_cap=label, engine_id=last.engine_id.to_numpy(), actual_RUL=truth, predicted_RUL=p)))
    official = pd.DataFrame(official_rows)
    official.to_csv(RUN / "official_test_metrics.csv", index=False)
    pd.concat(official_predictions, ignore_index=True).to_csv(RUN / "official_test_predictions.csv", index=False)
    comparison = official.pivot(index="dataset", columns="method", values="raw_RMSE")
    comparison["RMSE_reduction_percent"] = 100 * (
        comparison.fixed100-comparison.validation_selected)/comparison.fixed100
    comparison.to_csv(RUN / "official_test_comparison.csv")
    display(official.round(3))
    display(comparison.round(3))
    print("Positive reduction = improvement over the matched fixed-100 reference.")
else:
    print("Validation complete. Official evaluation disabled.")


## What to report
Report selected training caps from `selected_caps.csv`, validation RMSE mean and split standard deviation, and official **raw** RMSE/MAE/R² for the selected and fixed-100 models. Positive signed bias means overestimated remaining life. Report a test deterioration if it occurs; validation improvement does not guarantee test improvement.

Cap sensitivity can reveal a trade-off between near-failure accuracy and predicting long lifetimes. This is a cap experiment with a fixed feature pipeline, not an exhaustive search for the best model. It does not establish real-engine maintenance safety. Keep previous experiment results alongside this output.
